# **Explainable AI Tools for Deep Learning-Based Glucose Forecasting**

This notebook demonstrates the explainability utilities provided by the `gluex` package using a pre-trained deep learning model and synthetic example data. It presents a concise, executable workflow that generates impulse response visualizations, partial dependence plots (PDPs), and counterfactual explanations. These tools help verify whether the model’s behavior is physiologically plausible—specifically, whether increasing insulin doses leads to lower predicted blood glucose (BG) levels, and whether larger carbohydrate (CHO) intakes lead to higher predicted BG levels.

### What you will see
- Load a pre-trained model from the `Models/` folder and construct mock `x_test` inputs (replace with your real sequences for meaningful results).
- Produce an impulse response plot (data-independent) and save an interactive HTML in `Saved_data/ImpulseResponse/`.
- Generate interactive partial dependence plots on example inputs and save them to `Saved_data/PDP/`.
- Run counterfactual analysis on example inputs and save results to `Saved_data/Counterfactuals.pkl`

### Quick start / requirements
- A working model saved under `Models/` (default example loads `Models/PhyNet.keras`).
- Model input shape: `(seq_length, n_features)` (default: `(18, 3)`).
- Model output: a horizon-sized vector `(18,)` matching PH = 90 min / 5 min = 18 steps.
- Feature order expected by the notebook: `[blood_glucose, insulin, carbohydrate]`.

### Reproducibility & notes
- The first code cell sets a fixed random seed for reproducible mock examples.
- Replace the synthetic `x_train` / `x_test` arrays with your real preprocessed sequences before running PDPs or counterfactuals — otherwise results are illustrative only.
- PDPs and counterfactual analyses depend on input data and will therefore differ from the paper (the T1DEXI dataset used in the paper is not publicly available; it can be requested from the dataset owners).
- The impulse response analysis is data-independent. Using PhyNet with the same impulse amplitudes as in the paper will reproduce the published figure.

### Saved outputs
- `Saved_data/ImpulseResponse/`: interactive impulse response plots
- `Saved_data/PDP/`: partial dependence plots
- `Saved_data/Counterfactuals.pkl`: counterfactual examples and metrics

### Next steps
Run the notebook cells in order. To obtain meaningful explanations, prepare and pass your real preprocessed sequences (same feature order and shapes).

# 1. Notebook Setup

This section handles the necessary imports and sets up the parameters for the notebook.

In [19]:
from keras.utils import set_random_seed
import numpy as np
from gluex import *
import pickle
import os
import pandas as pd

# set seed for reproducibility
set_random_seed(0)  
print("Imports complete.")

#create folders to save results if they do not exist
createFolder = lambda directory: os.makedirs(directory) if not os.path.exists(directory) else None
createFolder("Saved_data/ImpulseResponse/")
createFolder("Saved_data/PDP/")

#set notebook parameter (used for visualization of results)
Ts = 5  # sampling interval of the data [min]

Imports complete.


# 2. Model Loading and Mock Data Generation

Here, we load a pre-trained model and generate synthetic data for demonstration purposes.

### Model Loading

In [20]:
# Load models using blood glucose, insulin, and carbs as inputs

model_name="PhyNet" #choose from {"CNN", "LSTM", "CNN-LSTM", "TCN", "MLP", "CNN-Transformer", "PhyNet"} 

#handle different file extensions for the models (CNN-Transformer uses .tf, others use .keras)
model_path=f"Models/{model_name}.tf" if model_name=="CNN-Transformer" else f"Models/{model_name}.keras"
model=load_model(model_path)
print(f"{model_name} loaded.")

# Get model input sequence length and prediction horizon in number of timepoints
if model_name=="CNN-Transformer": 
    seq_length,PH=model.model.input_shape[0][1], model.model.output_shape[1]
else:
    seq_length,PH=model.model.input_shape[1], model.model.output_shape[1]

print(f"Input sequence length: {seq_length*Ts} minutes \nMaximum prediction horizon: {PH*Ts} minutes")


PhyNet loaded.
Input sequence length: 90 minutes 
Maximum prediction horizon: 90 minutes


>**Note:** In this example, the input sequence length and prediction horizon are equal, but this may vary for different models.

### Mock Data Generation

In [21]:
#Create mock data for PDP and counterfactual analysis
num_examples_train=100000
num_examples_test=10000

#used to fit the counterfactual explainer
x_train, y_train = create_mock_data(num_examples_train, seq_length, PH)

#used for the PDPs and counterfactual explanations
x_test, y_test = create_mock_data(num_examples_test, seq_length, PH)

print("Mock data created")

Mock data created


This data is generated to simulate a time series for blood glucose management. Here's a breakdown of how each component is created:

- **Blood Glucose (BG):** Each sequence starts with a random baseline BG value between 40 and 200 mg/dL. A random walk is then simulated by adding cumulative random noise (from a normal distribution) at each time step to mimic physiological fluctuations.

- **Insulin:** Insulin administrations are modeled as sparse events. Assuming 4 injections per day, specific time points are randomly selected to receive an insulin dose. The dose amounts are drawn from a log-normal distribution to reflect realistic dosing patterns.

- **Carbohydrates (CHO):** Similar to insulin, carbohydrate intake (meals) is modeled as sparse events, assuming 4 meals per day. The size of each meal (in grams) is also drawn from a log-normal distribution.

- **Output Data:** The target BG sequence for the prediction horizon is generated by continuing the random walk from the last BG value of the input sequence.

> **Note:** This process generates a synthetic dataset that does not fully capture the key characteristics of real-world glucose, insulin, and carbohydrate data, but it is sufficient for demonstrating the tools in this package.

# 3. Impulse Response Analysis

In this section, we examine the model's response to impulse perturbations in the exogenous inputs using a synthetic example. This allows for a straightforward evaluation of how the model reacts to insulin boluses or carbohydrate (CHO) intakes of varying magnitudes in a controlled setting.

The synthetic input is constructed as follows:
1.  **Blood Glucose (BG):** The BG signal is fixed to a constant value equal to the average CGM level in the training set (145 mg/dL).
2.  **Exogenous Inputs:** Both insulin and CHO series are set to zero throughout the input window.
3.  **Impulse:** At prediction time, a single impulse of amplitude $Z$ is introduced to one of the exogenous inputs (either insulin or CHO).

We vary $Z$ across multiple magnitudes to observe how increasing doses influence the model's predictions.

In [22]:
# Model response to impulses of insulin or carbs at prediction time 

impulses={'Insulin':[0,1.5,3.0,4.5,9], 'CHO':[0,15,30,45,90]} #impulse amplitudes (U for insulin, g for carbs)

plot=impulse_response(
    model, impulses=impulses, ins_cho_indx=[1,2],
    save_to=f"Saved_data/ImpulseResponse/{model_name}", verbose=0, Ts=Ts,
)

print(f"Saved impulsive response")
plot

Saved impulsive response


:Layout
   .Overlay.I  :Overlay
      .NdOverlay.I  :NdOverlay   [CHO intake]
         :Curve   [time]   (value)
      .NdOverlay.II :NdOverlay   [Variable]
         :Scatter   [time]   (value)
   .Overlay.II :Overlay
      .NdOverlay.I  :NdOverlay   [Insulin bolus]
         :Curve   [time]   (value)
      .NdOverlay.II :NdOverlay   [Variable]
         :Scatter   [time]   (value)

> **Note:** Since impulse response analysis is data-independent, using the **PhyNet** model with the amplitudes specified in the paper will reproduce the reported results. However, the **LSTM** model provided in this repository requires raw insulin and carbohydrate (CHO) inputs, as we cannot disclose the proprietary filters used for Carbohydrates On Board (COB) and Insulin On Board (IOB) preprocessing. Consequently, the LSTM impulse response figure generated here will differ from the one in the paper.

# 4. Partial Dependence Plots

To assess sensitivity of the models to feature perturbations on real instances, we employed PDPs, which aggregate model responses over a distribution of real-world inputs and cover the entire range of feature values observed in the training data (although in this notebook we rely on mock data for this analysis). Specifically, PDPs were constructed following this procedure: (1) sampling a subset of test data to reduce computational cost; (2) zeroing all exogenous inputs over the input windows; (3) simulating boluses of amplitude $Z$ at prediction time for a selected feature (either insulin or CHO); (4) generating predictions with and without the perturbation; and (5) for each amplitude $Z$, plotting median and interquartile range (IQR) of the resulting prediction differences at the chosen PH (30 minutes in this study).

In [23]:
# Partial Dependence Plot for insulin and carbs
ins_cho_max_demo = [10,100] # [Insulin max [U], Carbs max [g]], use a reduced amplitude range (ins_cho_max) for a fast demo
PH_pdp = 30  # [min] outcome used for PDP

_,_,plot=PartialDependencePlot(
    model, x_test, batch_size=256, ins_cho_max=ins_cho_max_demo, sample_size=500, 
    save_to=f"Saved_data/PDP/{model_name}", PH_idx=PH_pdp//Ts-1)
plot

:Layout
   .Overlay.I  :Overlay
      .Curve.Median :Curve   [CHO]   (median)
      .Area.I       :Area   [CHO]   (25th percentile,75th percentile)
      .NdOverlay.I  :NdOverlay   [Variable]
         :Curve   [CHO]   (value)
   .Overlay.II :Overlay
      .Curve.Median :Curve   [Ins]   (median)
      .Area.I       :Area   [Ins]   (25th percentile,75th percentile)
      .NdOverlay.I  :NdOverlay   [Variable]
         :Curve   [Ins]   (value)

> **Note:** This plot will be completely different from those in the paper, as the data for the explanations used is different. The T1DEXI dataset used in the paper is not publicly available, but access can be requested through the following link: [https://doi.org/10.25934/PR00008428](https://doi.org/10.25934/PR00008428).

# 5. Counterfactual Analysis

CF explanations provide insights into why a model made a particular prediction by generating hypothetical instances that are similar to the original input but yield different prediction outcomes. 

### Select and prepare data for Counterfactual Explanations

Building on the [DiCE Python package](https://github.com/interpretml/DiCE), we implemented a genetic algorithm to minimize a cost function comprising three components: an interval-insensitive loss on the predictions, which encourages modifications that place the prediction within the target range (where the loss is zero); a *proximity* term, which measures how far the counterfactual is from the original input; and a *sparsity* term, which quantifies the number of features that differ from the original input.

In the next cell we select the examples to explain. Specifically, we chose the examples where the last CGM value was within the target range and the model predicted an undesirable outcome (glucose levels 30 minutes ahead outside the 80–180 mg/dL).

In [25]:
num_examples = 5  # Number of examples to explain. A smaller number will result in a faster demonstration.
PH_dice = 30 # Prediction horizon in minutes, used as the outcome for counterfactual explanations.
desired_range = (80,180)  # Desired blood glucose (BG) range for the outcome of the counterfactual.

# Select examples for explanation based on the model's predictions.
# We will only select examples that are predicted to be either hypo- or hyperglycemic,
# and for which the last CGM reading was within the desired range.
y_pred_tot = model.predict(x_test, batch_size=2**8, verbose=2) 
y_pred_tot = y_pred_tot[:,PH_dice//Ts-1] # Select the prediction at the PH_dice time step.

x_test_array=np.array(x_test)
last_CGM = x_test_array[:,-1,0]

# Find indices of examples predicted to be hypoglycemic.
indexes_hypo = np.where(np.logical_and(y_pred_tot<desired_range[0], last_CGM>desired_range[0]))[0] 
# Find indices of examples predicted to be hyperglycemic.
indexes_hyper = np.where(np.logical_and(y_pred_tot>desired_range[1], last_CGM<desired_range[1]))[0] 

# Randomly select up to 5 hypoglycemic examples to explain.
ind1=np.random.choice(indexes_hypo, min(num_examples//2,len(indexes_hypo)), replace=False) 
# Select the remaining examples from the hyperglycemic pool to reach a total of `num_examples`.
ind2=np.random.choice(indexes_hyper, min(num_examples-len(ind1),len(indexes_hyper)), replace=False)

num_hypo_hyper={'hypo': len(ind1) , 'hyper': len(ind2)}

indexes = np.concatenate((ind1,ind2))
print(f"hypo_examples to explain: {num_hypo_hyper['hypo']} \nhyper_examples to explain: {num_hypo_hyper['hyper']}")

# Extract the selected examples for explanation.
x_explain = [x_test[i] for i in indexes] 
y_explain = y_test[indexes,:] 

# Prepare the data for counterfactual generation using the DiCE format.
features_label = ["BG","Insulin","CHO"] # Defines the order of features in the input data.
data_model, query_instances_og, output_values_og,feature_names = DataPreparationDice(x_train,y_train, x_explain, y_explain,
                                                                                PH_dice=PH_dice, Ts=Ts, features_label=features_label)

40/40 - 2s - 2s/epoch - 52ms/step
hypo_examples to explain: 0 
hyper_examples to explain: 5


### Generate Counterfactual Explanations

In [26]:
#select features to keep constant during counterfactual search
fixed_features = [f"{x}_{i}" for x in ["Insulin","CHO"] for i in range(0, 75, 5) ] + \
    [f"BG_{i}" for i in range(0, 90, 5)] #features to keep constant
# select features to vary during counterfactual search: only insulin and carbs for the last 3 timepoints (15 minutes)
features_to_vary = [feat for feat in feature_names if not any(feat in x for x in fixed_features)]

try:
    cfs_and_queries, cf_metrics=counterfactual_analysis(data_model, model, query_instances_og, total_cfs = 1,
                                                        features_to_vary=features_to_vary, desired_range=desired_range,
                                                        posthoc_sparsity_algorithm="binary" if 'PhyNet' in model_name else "linear", #linear search can be used only if input/output relationship is monotonic
                                                        posthoc_sparsity_param=1, PH_dice=PH_dice)
except Exception as e:
    print(f"Counterfactual generation failed: {e}")
    cfs_and_queries = None
    cf_metrics = None

#save the counterfactuals and metrics 
data_to_save = {
    "cfs_and_queries": cfs_and_queries,
    "cf_metrics": cf_metrics
}

with open(f"Saved_data/Counterfactuals.pkl", "wb") as f:
    pickle.dump(data_to_save, f)

Generating counterfactuals for 5 examples...


  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:05<00:22,  5.58s/it]

 40%|████      | 2/5 [00:10<00:16,  5.35s/it]

 60%|██████    | 3/5 [00:16<00:10,  5.40s/it]

 80%|████████  | 4/5 [01:08<00:24, 24.05s/it]

100%|██████████| 5/5 [01:14<00:00, 14.82s/it]


Calculating physiological plausibility and metrics...
Counterfactual analysis completed.


### Analyze and Summarize Counterfactual Metrics

In [27]:
# utility to avoid division by zero
handle_zero_division = lambda x: x if x!=0 else np.nan

# assumes that hypo counterfactuals are first in the list
indx_hypo=np.array([i for i in range(num_hypo_hyper['hypo'])])
indx_hyper=np.array([i for i in range(num_hypo_hyper['hypo'],num_hypo_hyper['hypo']+num_hypo_hyper['hyper'])]) #because hyper counterfactuals start after hypo ones
indexes = {'hypo': indx_hypo, 'hyper': indx_hyper}


#initialize variable to store results
plausibility_metrics,summary_metrics,metrics_data = pd.DataFrame(), pd.DataFrame(), {}
for key in indexes.keys():
    # Summarize proximity metrics as median and interquartile range
    proximity_df = pd.DataFrame(cf_metrics['proximity'])
    proximity_median = proximity_df.loc[indexes[key],:].median()
    proximity_25_75= proximity_df.loc[indexes[key],:].quantile([.25,.75]) 

    # Summarize sparsity metrics as median and interquartile range
    sparsity_df = pd.DataFrame(cf_metrics['sparsity'])*100 #convert to percentage
    sparsity_median = sparsity_df.loc[indexes[key],:].median()
    sparsity_25_75= sparsity_df.loc[indexes[key],:].quantile([.25,.75]) 

    # Store the proximity and sparsity metrics in a dataframe
    metrics_data = {
    'Proximity CHO [g]': f"{proximity_median['CHO']:.2f} [{proximity_25_75.loc[0.25,'CHO']:.2f}-{proximity_25_75.loc[0.75,'CHO']:.2f}]",
    'Sparsity CHO [%]': f"{sparsity_median['CHO']:.2f} [{sparsity_25_75.loc[0.25,'CHO']:.2f}-{sparsity_25_75.loc[0.75,'CHO']:.2f}]",
    'Proximity Insulin [U]': f"{proximity_median['Insulin']:.2f} [{proximity_25_75.loc[0.25,'Insulin']:.2f}-{proximity_25_75.loc[0.75,'Insulin']:.2f}]",
    'Sparsity Insulin [%]': f"{sparsity_median['Insulin']:.2f} [{sparsity_25_75.loc[0.25,'Insulin']:.2f}-{sparsity_25_75.loc[0.75,'Insulin']:.2f}]",
    }
    summary_metrics = pd.concat([summary_metrics, pd.DataFrame(metrics_data,index=[key])])
    
    # Store plausibility metrics as percentages in a DataFrame with hypo/hyper as index
    plaus_dict={k.replace(f"_{key}"," [%]"):round(v/handle_zero_division(num_hypo_hyper[key])*100,1) for k,v in cf_metrics['plausibility'].items() if key in k} #key refers to 'hypo' or 'hyper'
    plausibility_metrics=pd.concat([plausibility_metrics, pd.DataFrame(plaus_dict,index=[key])])
   
# Concatenate plausibility and summary metrics
tot_metrics = pd.concat([plausibility_metrics, summary_metrics], axis=1)
tot_metrics.loc[:,'Num Instances']=num_hypo_hyper #number of instances
tot_metrics = tot_metrics[['Num Instances'] + [col for col in tot_metrics.columns if col != 'Num Instances']] #reorder columns

tot_metrics

,Num Instances,Ambiguous [%],Not found [%],Plausible [%],Implausible [%],Proximity CHO [g],Sparsity CHO [%],Proximity Insulin [U],Sparsity Insulin [%]
hypo,0,NaN,NaN,NaN,NaN,nan [nan-nan],nan [nan-nan],nan [nan-nan],nan [nan-nan]
hyper,5,0.0,20.0,80.0,0.0,0.00 [0.00-0.00],100.00 [100.00-100.00],0.19 [0.16-0.56],66.67 [66.67-66.67]


> **Note:** The results will be completely different from those in the paper, as the data for the explanations used is different. The T1DEXI dataset used in the paper is not publicly available, but access can be requested through the following link: [https://doi.org/10.25934/PR00008428](https://doi.org/10.25934/PR00008428).

To move beyond instance-level explainability and provide a global perspective on the generated CFs, we analyzed insulin and CHO modifications separately by computing two metrics for each: 

- **proximity**, measured as the L1 norm of the difference between a CF and its corresponding original example for that exogenous input;
- **sparsity**, quantified as the percentage of modifiable timepoints that were altered for that exogenous input (i.e., the L0 norm expressed as a percentage).

These metrics were first computed for each counterfactual individually and then summarized across counterfactuals using the median and interquartile range (IQR). More importantly, we evaluated whether the counterfactual modifications were physiologically plausible:

- **For hyperglycemic instances**, a counterfactual was deemed:
  - *Plausible* if it involved an increase in insulin and/or a decrease in CHO
  - *Implausible* if it involved an increase in CHO and/or a decrease in insulin

- **For hypoglycemic instances**, a counterfactual was considered:
  - *Plausible* if it involved an increase in CHO and/or a decrease in insulin
  - *Implausible* if it involved an increase in insulin and/or a decrease in CHO

- Counterfactuals were labeled as *ambiguous* if they involved simultaneous changes in the same direction across both features, as such modifications produce unpredictable effects.

- Finally, in some cases, the genetic algorithm was unable to produce a counterfactual leading to a predicted glucose value inside the target range; these instances were labeled as *not found*.